# Protein SideChainet data loder


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import os 
import numpy as np
from tqdm import tqdm

In [2]:
# Parameters for data loading
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Use GPU is avaliable
if(DEVICE.type == "cuda" or DEVICE.type == "mps"):
    torch.cuda.empty_cache()
    CUDA = True
else:
    CUDA = False
    
DATA_DIR = '../data/casp12_data_30/'
DEBUG = False # If true, load only 10 samples
BATCH_SIZE = 1
NUM_WORKERS = 0 

In [3]:
# Dataset parameters
DEBUG_SAMPLE_SIZE = 10
NANO_TO_ANGSTROM = 10**(-10)
GAUSSIAN_COEF = -0.008*1e1

In [7]:
class SidChainDS(Dataset):
    """Protein dataset."""
    def __init__(self, data_path ,set_type,debug):
        """
            Initialize the dataset
        Args:
            data_path (str): data path
            set_type (str): 'train','test' or 'valid'
        """
        self.data_path = data_path
        self.set_type = set_type
        self.data_dir = []
        if(set_type == 'valid'):
            self.folders  =[data_path+val_path for val_path in ['valid-10/','valid-20/','valid-30/','valid-40/','valid-50/']]
            for folder in self.folders:
                self.data_dir.extend([folder+f for f in os.listdir(folder) if os.path.isdir(os.path.join(folder, f))])
        else:
            self.data_dir = [os.path.join(data_path+set_type, f) for f in os.listdir(data_path+set_type) if os.path.isdir(os.path.join(data_path+set_type, f))]   
        if(debug):
            self.data_dir = self.data_dir[:DEBUG_SAMPLE_SIZE]
            
    def __getitem__(self, index):
        
        item_path = self.data_dir[index]
        # load data
        id = torch.load(item_path + '/id.pt')
        crd_backbone = torch.tensor(torch.load(item_path + '/crd_backbone.pt'),dtype=torch.get_default_dtype()) #backbone coordinates N,Calpha,C
        mask = torch.load(item_path + '/mask.pt')
        # change to 1,0 mask
        mask = torch.tensor(np.where(np.array(list(mask))=='+',1,0))
        seq_one_hot = torch.load(item_path + '/seq_one_hot.pt')
        seq = torch.load(item_path + '/seq.pt')
        proT5_emb = torch.load(item_path + '/proT5_emb.pt')
        # proT5_emb = torch.zeros((len(seq),1024)) # for testing
        ang = torch.tensor(torch.load(item_path + '/ang.pt'))
        ang_backbone = torch.clone(ang)[:,:3] #angles for the backbone phi, psi, omega
        # Add Cbeta atom to the coordinates
        crd_backbone = self.add_cb(crd_backbone)
        crd_backbone = crd_backbone * NANO_TO_ANGSTROM # Convert to angstrom
        
        return id, crd_backbone, mask, seq_one_hot, seq,ang_backbone, ang, proT5_emb

    def __len__(self):
        return len(self.data_dir)

  
    def add_cb(self,crd_coords):
        """
        Add the Cbeta atom to the coordinates
        Args:
            crd_coords (tensor): tensor of shape [n_residues,3,3]

        Returns:
            crd_coords: tensor shape [n_residues,4,3]
        """
        # Get the coordinates of the backbone atoms
        N, CA, C = crd_coords[:, 0], crd_coords[:, 1], crd_coords[:, 2]
        # CB = CA + c1*(N-CA) + c2*(C-CA) + c3* (N-CA)x(C-CA)
        CAmN = N - CA
        # CAmN = CAmN / torch.sqrt(CAmN ** 2).sum(dim=2, keepdim=True)
        CAmC = C - CA
        # CAmC = CAmC / torch.sqrt(CAmC ** 2).sum(dim=2, keepdim=True)
        ANxAC = torch.cross(CAmN, CAmC, dim=1)

        A = torch.cat((CAmN.reshape(-1, 1), CAmC.reshape(-1, 1), ANxAC.reshape(-1, 1)), dim=1)
        c = torch.tensor([0.5507, 0.5354, -0.5691]) / 100  # torch.tensor([1.1930, 1.2106, -2.7906]) #
        b = (A @ c).reshape(-1,3)
        CB = CA - b
      
        # Add Cbeta coordinates to existing coordinates array
        crd_coords = torch.cat((crd_coords, CB.unsqueeze(1)), dim=1)
        return crd_coords
    
    
    

In [8]:
# Create data loader from sidechainnet dataset  
train_loader= DataLoader(SidChainDS(data_path=DATA_DIR,set_type='train', debug=DEBUG), batch_size=BATCH_SIZE, shuffle=True,
                                    num_workers=NUM_WORKERS,
                                    pin_memory=CUDA)
valid_loader= DataLoader(SidChainDS(data_path=DATA_DIR,set_type='valid', debug=DEBUG), batch_size=BATCH_SIZE, shuffle=True,
                                    num_workers=NUM_WORKERS,
                                    pin_memory=CUDA)

test_loader= DataLoader(SidChainDS(data_path=DATA_DIR,set_type='test', debug=DEBUG), batch_size=BATCH_SIZE, shuffle=True,
                                    num_workers=NUM_WORKERS,
                                    pin_memory=CUDA)

Show example of the data

In [13]:
with tqdm(train_loader, unit="batch") as tepoch:
         for i, data in enumerate(tepoch):
            id, crd_backbone, mask, seq_one_hot, seq,ang_backbone, ang, proT5_emb = data
            print("Id : ", id)
            print("crd_backbone shape: ", crd_backbone.shape)
            print("mask shape: ", mask.shape)
            print("seq_one_hot shape: ", seq_one_hot.shape)
            print("seq : ", seq)
            print("ang_backbone shape: ", ang_backbone.shape)
            print("ang shape: ", ang.shape)
            print("proT5_emb shape: ", proT5_emb.shape)
            # move to GPU
            crd_backbone = crd_backbone.to(DEVICE)
            mask = mask.to(DEVICE)
            seq_one_hot = seq_one_hot.to(DEVICE)
            proT5_emb = proT5_emb.to(DEVICE)
            break

  0%|          | 0/25213 [00:00<?, ?batch/s]

Id :  ('3N2C_1_A',)
crd_backbone shape:  torch.Size([1, 423, 4, 3])
mask shape:  torch.Size([1, 423])
seq_one_hot shape:  torch.Size([1, 423, 20])
seq :  ('MSLTITVLQGGNVLDLERGVLLEHHHVVIDGERIVEVTDRPVDLPNAQAIDVRGKTVMPGFIDCHVHVLASNANLGVNATQPNILAAIRSLPILDAMLSRGFTSVRDAGGADWSLMQAVETGLVSGPRIFPSGKALSQTGGHGDFRPRGDLLEPCSCCFRTGAIARVVDGVEGVRLAVREEIQKGATQIKIMASGGVASPTDPIANTQYSEDEIRAIVDEAEAANTYVMAHAYTGRAIARAVRCGVRTIEHGNLVDEAAAKLMHEHGAFVVPTLVTYDALAKHGAEFGMPPESVAKVASVQQKGRESLEIYANAGVKMGFGSDLLGEMHAFQSGEFRIRAEVLGNLEALRSATTVAAEIVNMQGQLGVIAVGAIADLVVLDGNPLEDIGVVADEGARVEYVLQRGTLVKRQAAGREGHHHHHH',)
ang_backbone shape:  torch.Size([1, 423, 3])
ang shape:  torch.Size([1, 423, 12])
proT5_emb shape:  torch.Size([1, 423, 1024])


get graph function:
The function sets masked nodes to zero distance matrix and adds the embedding

In [14]:
def get_graph(x, one_hot, emb, mask):
    """Get graph representation of protein"""
    D = get_dist_matrix(x) # N,N,16
    D = torch.relu(torch.exp( GAUSSIAN_COEF*D**2))
    # remove masks values
    mask_index = torch.where(mask == 0)
    D[mask_index[0],:,:] = 0
    D[:,mask_index[0],:] = 0
    # get bonded features
    Fb = get_bonded_features(D) # N,32
    # sum over the atoms
    D = D.sum(dim=1) #N,16
    D = F.normalize(D,p=2,dim=0)
    Fh = torch.cat([D,Fb,emb,one_hot],dim=1) #N,16+32+emb_size
    
    return Fh

def get_bonded_features(D):
    """Get bonded features from distance matrix"""
    n_range = torch.arange(D.shape[0])
    # get above and below diagonal
    f1, f2 = D[n_range[:-1], n_range[1:]], D[n_range[1:], n_range[:-1]]
    # pad with zeros
    zero_row = torch.zeros((1, D.shape[-1])).to(D.device)
    f1 = torch.cat([f1, zero_row], dim=0)
    f2 = torch.cat([zero_row, f2], dim=0)
    # concat features
    Fb = torch.cat([f1, f2], dim=-1)
    return Fb # N,32

def get_dist_matrix(Xd):
    """
    Return the node distence matrix
    Args:
        Xd (tensor):X embeded [n_nodes ,num_atoms=4,new_cords_size]
    Returns:
        tensor : [n_nodes,n_nodes ,atom_dist=16] tensor
    """
    N_residu, N_atoms, coords_size = Xd.shape
    Xd = Xd.reshape(N_residu*N_atoms, coords_size)
    D = torch.cdist(Xd, Xd, p=2)
    D = D.reshape(N_residu, N_atoms, N_residu, N_atoms)
    D = torch.swapaxes(D,1,2)
    D = D.reshape(N_residu, N_residu, N_atoms*N_atoms)
    return D

In [15]:
# squeeze all the inputs
crd_backbone,seq_one_hot, proT5_emb, mask = crd_backbone.squeeze(),seq_one_hot.squeeze(), proT5_emb.squeeze(), mask.squeeze()
X = get_graph(crd_backbone,seq_one_hot, proT5_emb, mask)
X.shape # N,16+32+emb_size

/var/folders/jc/8jq74lx123q5kwlm2vgv1lz00000gn/T/ipykernel_4066/889633902.py:6: UserWarning: The operator 'aten::nonzero' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1670525699189/work/aten/src/ATen/mps/MPSFallback.mm:11.)
  mask_index = torch.where(mask == 0)


torch.Size([423, 1092])